# 30. Everything except LOW, which quietly excludes the unknowns

**Track:** pyspark | **Difficulty:** easy | **Tags:** filter, not equal, three-valued logic, nulls

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("030-exclude-value-drops-nulls")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "tickets": """
UEFSMRUEFbAbFfoNTBW2AxUAEgAA2A0IiRMABQEAig0IAIsNCACMDQgAjQ0IAI4NCACPDQgAkA0IAJENCACSDQgAkw0IAJQNCACV
DQgAlg0IAJcNCACYDQgAmQ0IAJoNCACbDQgAnA0IAJ0NCACeDQgAnw0IAKANCAChDQgAog0IAKMNCACkDQgApQ0IAKYNCACnDQgA
qA0IAKkNCACqDQgAqw0IAKwNCACtDQgArg0IAK8NCACwDQgAsQ0IALINCACzDQgAtA0IALUNCAC2DQgAtw0IALgNCAC5DQgAug0I
ALsNCAC8DQgAvQ0IAL4NCAC/DQgAwA0IAMENCADCDQgAww0IAMQNCADFDQgAxg0IAMcNCADIDQgAyQ0IAMoNCADLDQgAzA0IAM0N
CADODQgAzw0IANANCADRDQgA0g0IANMNCADUDQgA1Q0IANYNCADXDQgA2A0IANkNCADaDQgA2w0IANwNCADdDQgI3hMABQEA3w0I
AOANCADhDQgA4g0IAOMNCADkDQgA5Q0IAOYNCADnDQgA6A0IAOkNCADqDQgA6w0IAOwNCADtDQgA7g0IAO8NCADwDQgA8Q0IAPIN
CADzDQgA9A0IAPUNCAD2DQgA9w0IAPgNCAD5DQgA+g0IAPsNCAD8DQgA/Q0IAP4NCAD/DQgEABQlDwQAAQ0IAAINCAADDQgABA0I
AAUNCAAGDQgABw0IAAgNCAAJDQgACg0IAAsNCAAMDQgADQ0IAA4NCAAPDQgAEA0IABENCAASDQgAEw0IABQNCAAVDQgAFg0IABcN
CAAYDQgAGQ0IABoNCAAbDQgAHA0IAB0NCAAeDQgAHw0IACANCAAhDQgAIg0IACMNCAAkDQgAJQ0IACYNCAAnDQgAKA0IACkNCAAq
DQgAKw0IACwNCAAtDQgALg0IAC8NCAAwDQgAMQ0IADINCAAzDQgANA0IADUNCAA2DQgANw0IADgNCAA5DQgAOg0IADsNCAA8DQgA
PQ0IAD4NCAA/DQgAQA0IAEENCABCDQgAQw0IAEQNCABFDQgARg0IAEcNCABIDQgASQ0IAEoNCABLDQgATA0IAE0NCABODQgATw0I
AFANCABRDQgAUg0IAFMNCABUDQgAVQ0IAFYNCABXDQgAWA0IAFkNCABaDQgAWw0IAFwNCABdDQgAXg0IAF8NCABgDQgAYQ0IPGIU
AAAAAAAAYxQAAAAAAAAVABXSAxXaAywVtgMVEBUGFQYcGAhjFAAAAAAAABgIiRMAAAAAAAAWACgIYxQAAAAAAAAYCIkTAAAAAAAA
EREAAADpAfDoAwAAALYDAQg5AAECAwQFBgcICQoLDA0ODxAREhMUFRYXGBkaGxwdHh8gISIjJCUmJygpKissLS4vMDEyMzQ1Njc4
OTo7PD0+P0BBQkNERUZHSElKS0xNTk9QUVJTVFVWV1hZWltcXV5fYGFiY2RlZmdoaWprbG1ub3BxcnN0dXZ3eHl6e3x9fn+AgYKD
hIWGh4iJiouMjY6PkJGSk5SVlpeYmZqbnJ2en6ChoqOkpaanqKmqq6ytrq+wsbKztLW2t7i5uru8vb6/wMHCw8TFxsfIycrLzM3O
z9DR0tPU1dbX2NnaAAAAAAAVBBWQARWSAUwVDBUAEgAASGAJAAAAbG9naXN0aWNzCAAAAHBheW1lbnRzAQyoYWNjb3VudHMKAAAA
b25ib2FyZGluZwcAAABjYXRhbG9nBgAAAHNlYXJjaBUAFboBFcABLBW2AxUQFQYVBhw2ACgGc2VhcmNoGAhhY2NvdW50cxERAAAA
XfBcAwAAALYDAQM5SCJgIakABJQsZJAQkgImQQlASAKlAkIhDNJJVASIERRoVJAsCSgkiAgxZSCAogoqCggKTKCEE8EEaSiCQoAx
IakRgCookRApk6hRIFFyUUJCJAAAFQQVRhVKTBUIFQASAAAjiAYAAABNRURJVU0DAAAATE9XBAAAAEhJR0gGAAAAVVJHRU5UFQAV
oAEVpgEsFbYDFRAVBhUGHDZoKAZVUkdFTlQYBEhJR0gREQAAAFDwTyAAAAAtzX9+f+/qOd1n39cf9/t/lX3f3/zk1xABBXL3FgED
3gIrJBIUdhQUQVeAUAIsZcVVj1ESVYUlgdghGqTFUQWFEpk9VhlVRyGRiFUMFQQVJBUoTBUEFQASAAASRAQAAABPUEVOBgAAAENM
T1NFRBUAFUwVUCwVtgMVEBUGFQYcNgAoBE9QRU4YBkNMT1NFRBERAAAAJpQDAAAAtgMBAS2WqhiDcokaNauKAvLCuUuACrASZIm6
GAAJQ1CpHhUEGVw1ABgGc2NoZW1hFQgAFQQlAhgJdGlja2V0X2lkABUMJQIYBHRlYW0lAEwcAAAAFQwlAhgIcHJpb3JpdHklAEwc
AAAAFQwlAhgGc3RhdHVzJQBMHAAAABa2AxkcGUwmABwVBBk1AAYQGRgJdGlja2V0X2lkFQIWtgMWqCAW+hImpA4mCBwYCGMUAAAA
AAAAGAiJEwAAAAAAABYAKAhjFAAAAAAAABgIiRMAAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYAtgMAAAAmABwVDBk1AAYQ
GRgEdGVhbRUCFrYDFsIDFsoDJrQUJoITHDYAKAZzZWFyY2gYCGFjY291bnRzEREAGSwVBBUAFQIAFQAVEBUCADwWyhsZBhkmALYD
AAAAJgAcFQwZNQAGEBkYCHByaW9yaXR5FQIWtgMW0gIW3AImshcmzBYcNmgoBlVSR0VOVBgESElHSBERABksFQQVABUCABUAFRAV
AgA8FqgLGQYZJmjOAgAAACYAHBUMGTUABhAZGAZzdGF0dXMVAha2AxbYARbgASbsGSaoGRw2ACgET1BFThgGQ0xPU0VEEREAGSwV
BBUAFQIAFQAVEBUCADwWrBAZBhkmALYDAAAAFpQoFrYDJggWgBsAGRwYDEFSUk9XOnNjaGVtYRj4Ai8vLy8veEFCQUFBUUFBQUFB
QUFLQUF3QUJnQUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNBQUlBQUFBQkFBSUFBQUFCQUFBQUFRQUFBQ2dBQUFBWUFBQUFEQUFBQUFF
QUFBQWdQLy8vd0FBQVFVUUFBQUFHQUFBQUFRQUFBQUFBQUFBQmdBQUFITjBZWFIxY3dBQXJQLy8vNmovLy84QUFBRUZFQUFBQUJ3
QUFBQUVBQUFBQUFBQUFBZ0FBQUJ3Y21sdmNtbDBlUUFBQUFEWS8vLy8xUC8vL3dBQUFRVVFBQUFBSEFBQUFBUUFBQUFBQUFBQUJB
QUFBSFJsWVcwQUFBQUFCQUFFQUFRQUFBQVFBQlFBQ0FBR0FBY0FEQUFBQUJBQUVBQUFBQUFBQVFJUUFBQUFKQUFBQUFRQUFBQUFB
QUFBQ1FBQUFIUnBZMnRsZEY5cFpBQUFBQWdBREFBSUFBY0FDQUFBQUFBQUFBRkFBQUFBQUFBQUFBPT0AGCBwYXJxdWV0LWNwcC1h
cnJvdyB2ZXJzaW9uIDI1LjAuMRlMHAAAHAAAHAAAHAAAAJ8DAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "030-exclude-value-drops-nulls"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

tickets = spark.read.parquet((_data_dir / "tickets.parquet").resolve().as_posix())
tickets.createOrReplaceTempView("tickets")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'team': '0f9cbc86350878bd31d9eadc6d039077350bb5e0254f4a22c71c7a9e8c5b8376',
                         'tickets_to_review': 'ec88748f73b7dd20add75f1abdc7dfe80455e0511a9f6ed268d5de626cebadde',
                         'untriaged': '638635186b63fbc83f1c94cf104c3a337bb685d77ac2a004f0b200630bb4cb1c'},
 'columns': [['team', 'string'], ['tickets_to_review', 'bigint'], ['untriaged', 'bigint']],
 'fingerprint': '56c77925b8b771cb32cb2fcbe988eb46c429f6da525aff094a5b601e3b5f8475',
 'order_matters': False,
 'row_count': 6}

check = make_check(EXPECTED)

## The situation

Each morning every support team gets the number of open tickets it has to look at.
Low-priority tickets are left out, because they are handled in a weekly batch.

New tickets arrive without a priority; someone on the team sets it during triage. The
morning number has been too low for weeks, and the tickets that are missing from it
are exactly the ones nobody has triaged yet.

## Input tables

### `tickets`

One row per ticket.

| Column | Type | Description |
|---|---|---|
| `ticket_id` | BIGINT | Unique per ticket. |
| `team` | STRING | The team that owns the ticket. |
| `priority` | STRING | `LOW`, `MEDIUM`, `HIGH` or `URGENT`. NULL until the ticket has been triaged. |
| `status` | STRING | `OPEN` or `CLOSED`. Never NULL. |

## Requirement

Return one row per team.

- `tickets_to_review` is the number of the team's `OPEN` tickets whose priority is
  not `LOW`. A ticket with no priority is not known to be low, so it counts.
- `untriaged` is the number of the team's `OPEN` tickets with no priority.
- Every team in `tickets` appears, with 0 where it has no such tickets.

## Expected output

One row per team.

| Column | Type | Description |
|---|---|---|
| `team` | STRING | |
| `tickets_to_review` | BIGINT | Open tickets that are not LOW, untriaged ones included. |
| `untriaged` | BIGINT | Open tickets with no priority. |

In [ ]:
tickets.show(5)

## Your answer

Use the DataFrame API. The tables are available as DataFrames: `tickets`.

In [ ]:
result = None  # Replace None with your DataFrame.

In [ ]:
check(result)